# Tutorial 6: Atlas-free 3D CNN inference

Use the same task/family/domain selectors for released checkpoints or a local standardized run. Mixed-baseline CNN weights are always the default; `variant="finetuned"` is an explicit opt-in.

In [ ]:
from neurovlm import AtlasFreeCNNDataProvider
from neurovlm.atlas_free_text import (
    AtlasFreeContrastiveCollator, AtlasFreeTextEmbeddingLookup,
)
from neurovlm.runtime import load_pipeline

DOMAIN = "pubmed"  # pubmed | nilearn | neurovault
provider = AtlasFreeCNNDataProvider(domain=DOMAIN, limit=2)
rows = [provider.test[index] for index in range(len(provider.test))]
batch = AtlasFreeContrastiveCollator(
    AtlasFreeTextEmbeddingLookup.published(), (36, 45, 38)
)(rows)

## Reconstruction

In [ ]:
autoencoder = load_pipeline(family="cnn", task="autoencoder")
reconstructed = autoencoder.reconstruct(batch["volume"])
autoencoder.metadata.as_dict(), reconstructed.shape

## Contrastive retrieval

In [ ]:
contrastive = load_pipeline(
    family="cnn", task="contrastive", domain=DOMAIN
)
similarity = contrastive.similarity(batch["volume"], batch["text_embedding"])
contrastive.metadata.as_dict(), similarity

## Text-to-brain generation

In [ ]:
generator = load_pipeline(
    family="cnn", task="text_to_brain", domain=DOMAIN
)
generated = generator.generate(batch["text_embedding"])
generator.metadata.as_dict(), generated.shape

## Local runs and explicit fine-tuning

Load a standardized run with `load_pipeline(..., from_run="runs/...")`. To request released domain-fine-tuned CNN weights, add `variant="finetuned"`. MLP uses the same loader with `family="mlp"`; the original `NeuroVLM` interface remains available for legacy high-level workflows.